<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/banners/banner_elearning_powerbi_guide.png" width="100%"/>
</div>

## 📖 Préambule

### À qui s'adresse ce guide ?

Ce notebook est ton **support de référence** pour construire le tableau de bord *EduTrack Analytics* dans Power BI Desktop. Il est conçu pour toi si :

- tu as déjà manipulé Excel et tu sais ce qu'est une formule,
- tu as installé Power BI Desktop sur ta machine,
- tu comprends ce qu'est une jointure entre deux tables.

### Comment lire ce guide

| Symbole | Ce qu'il indique |
|---|---|
| 🎯 | Ce que tu sauras faire à la fin de la section |
| 📘 | L'intuition métier ou technique avant de coder |
| 🔧 | Les clics, le code DAX, les paramètres exacts |
| ✅ | Comment vérifier que ton travail est correct |
| ⚠️ | L'erreur courante à éviter |
| 🚀 | L'optimisation ou la variante avancée |

### Le contexte métier

**EduTrack** est une plateforme e-learning panafricaine qui forme des apprenants à la data, au développement web, au marketing digital, au management et au leadership. Quatre métriques pilotent l'équipe pédagogique :

1. **Taux de complétion** — proportion d'apprenants qui terminent leur parcours (cible 40 %)
2. **Taux d'abandon** — proportion d'apprenants qui décrochent avant la fin
3. **CSAT** — satisfaction des apprenants (note de 1 à 5)
4. **Score de risque ML** — probabilité qu'un apprenant abandonne, calculée par un modèle entraîné sur l'engagement

Le dashboard que tu vas construire répond à 4 questions :

| Page | Question |
|---|---|
| 1 — Pédagogie & Acquisition | Comment se portent les complétions et d'où viennent les apprenants ? |
| 2 — Top 5 Parcours & Business | Quels parcours rapportent le plus et lesquels décrochent ? |
| 3 — Alerte Décrochage ML | Quels apprenants en cours sont à risque et quelle action prendre ? |
| 4 — Démographie | Qui sont nos apprenants (âge, pays, premium) ? |

## 🗺️ Sommaire détaillé

| Partie | Section | Durée |
|---|---|---|
| **I — Fondations** | Sources · Import CSV · Auto Date/Time | 30 min |
| **II — Modélisation** | Étoile · Calendrier · 5 relations · Marquage | 45 min |
| **III — Table `_Mesures`** | Création | 10 min |
| **IV — 62 mesures DAX** | Pédagogie 9 · Démographie 7 · Acquisition 2 · Paiement 2 · Business 4 · Variations 8 · Top 5 (15) · Alerte ML 15 | 3 h |
| **V — Colonnes calculées** | Tranche Age · Quadrant Instructeur · utilitaires | 40 min |
| **VI — Design** | Charte · Mockup PPTX → PNG | 30 min |
| **VII — 4 pages** | Pédagogie / Top Parcours / Alerte ML / Démographie | 1 h 15 |
| **VIII — Finitions** | Slicers · Navigation · Info-bulles | 20 min |
| **IX — Validation** | Checklist · Pièges · Storytelling · Annexes | 45 min |

**Total** ≈ **6 h 30** de travail effectif.

---
# I — Préparer les fondations

## 1.1 Comprendre les sources de données

### 📘 Concept clé — le « grain »

Une ligne de chaque table représente une entité bien identifiée. Avant d'écrire la moindre formule, complète mentalement : *« Une ligne de cette table = un(e) ____ »*.

### Inventaire des 5 sources

| Fichier | Grain | Rôle | Volumétrie |
|---|---|---|---|
| `apprenants.csv` | 1 apprenant unique | Dimension | 4 500 lignes |
| `parcours.csv` | 1 parcours de formation | Dimension | 12 lignes |
| `inscriptions_analytics.csv` | 1 inscription d'1 apprenant à 1 parcours, avec engagement et résultat | Fait principal | 6 456 lignes |
| `paiements.csv` | 1 paiement effectué (parfois plusieurs par parcours) | Fait | 8 703 lignes |
| `apprenants_risque_scores.csv` | 1 apprenant en cours, scoré par le modèle ML | Fait dérivé | 2 095 lignes |

### ⚠️ Piège fréquent — distinguer `inscriptions_analytics` et `apprenants_risque_scores`

`inscriptions_analytics` contient **tout l'historique**. `apprenants_risque_scores` ne contient que **les apprenants encore en cours** scorés par le ML. Les deux ne sont pas reliées directement.

- KPIs pédagogiques (complétion, abandon, CSAT) → **`inscriptions_analytics`**
- Compteurs d'alertes ML, scores de risque → **`apprenants_risque_scores`**
- Revenus, paniers moyens → **`paiements`**

## 1.2 Importer les CSV

### 📘 Concept clé — pourquoi GitHub raw plutôt que des fichiers locaux ?

Charger depuis une URL `raw.githubusercontent.com` te donne deux superpouvoirs :
1. **Reproductibilité** : tous les apprenants ont exactement la même donnée, à l'octet près.
2. **Mise à jour facile** : si on corrige une coquille dans le CSV, un simple *Actualiser* suffit, aucune ré-installation.

L'inconvénient : il faut une connexion internet au premier chargement. Une fois publié sur le service Power BI, le rapport peut être planifié pour rafraîchir tout seul.

### Les 6 URLs à utiliser

```
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/data/apprenants.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/data/parcours.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/data/inscriptions_analytics.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/data/paiements.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/data/apprenants_risque_scores.csv
```

> 💡 *Remplace par ton chemin de dépôt réel. Si les CSV sont en local pour l'instant, utilise `Obtenir les données → Texte/CSV` ; le reste de la procédure est identique.*

### ⚠️ Piège type pour `score_risque`

Si la colonne arrive en String avec point décimal, force-la en **Nombre décimal** dans Power Query, sinon `AVERAGE` plantera.

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/powerbi/tuto/01_powerquery_5_requetes.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 1.3 Désactiver l'Auto Date/Time

**Fichier → Options → Chargement des données → décocher Date/heure automatique**. Cela élimine les LocalDateTables fantômes qui ralentissent le modèle.

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/powerbi/tuto/02_options_auto_datetime.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# II — Modéliser les données

## 2.1 Le schéma en étoile

```
                    +------------------+
                    |   Calendrier     |
                    +--------+---------+
                             | 1
                             | N
    +----------------+   +---+---------------------+   +----------------+
    |   apprenants   |1  |                         |  1|   parcours     |
    +-------+--------+---+ inscriptions_analytics  +---+--------+-------+
            |          N |                         | N         |
            | 1          +-------------------------+           | 1
            |                                                  |
          N |                                                N |
    +-------+----------------+                       +---------+--------+
    | apprenants_risque_     |                       |   paiements      |
    | scores (ML)            |                       +------------------+
    +------------------------+

    +------------------+   (table de reference statique pour
    |  Pays Affichage  |    afficher proprement les pays Top 5
    +------------------+    + categorie Autres)
```

## 2.2 Créer la table Calendrier

**Modélisation → Nouvelle table** :

```dax
Calendrier = 
VAR _start = DATE(2023,1,1)
VAR _end   = DATE(2025,12,31)
RETURN
ADDCOLUMNS(
    CALENDAR(_start, _end),
    "Annee",         YEAR([Date]),
    "Mois_Num",      MONTH([Date]),
    "Mois_Nom",      FORMAT([Date], "mmm", "fr-FR"),
    "Mois_Nom_Long", FORMAT([Date], "mmmm", "fr-FR"),
    "Mois_court",    UPPER(LEFT(FORMAT([Date], "mmm", "fr-FR"), 3)),
    "Annee_Mois",    FORMAT([Date], "yyyy-MM"),
    "Trimestre",     "T" & QUARTER([Date]),
    "Semaine",       WEEKNUM([Date], 2),
    "Saison",        SWITCH(TRUE(),
                       MONTH([Date]) IN {12,1,2}, "Hiver",
                       MONTH([Date]) IN {3,4,5}, "Printemps",
                       MONTH([Date]) IN {6,7,8}, "Ete",
                       "Automne")
)
```

## 2.3 Établir les 5 relations

| # | De (1) | Clé | Vers (N) | Clé | Direction |
|---|---|---|---|---|---|
| 1 | `Calendrier` | `Date` | `inscriptions_analytics` | `date_inscription` | Single |
| 2 | `apprenants` | `apprenant_id` | `inscriptions_analytics` | `apprenant_id` | Single |
| 3 | `apprenants` | `apprenant_id` | `apprenants_risque_scores` | `apprenant_id` | Single |
| 4 | `parcours` | `parcours_id` | `inscriptions_analytics` | `parcours_id` | Single |
| 5 | `parcours` | `parcours_id` | `paiements` | `parcours_id` | Single |

**Vue Modèle → glisser la clé** de la table 1 vers la table N. Cardinalité 1:N, direction Simple.

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/powerbi/tuto/03_modele_etoile_relations.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

### ⚠️ Refuser systématiquement la bidirectionnelle proposée par Power BI — elle crée des ambiguïtés.

## 2.4 Marquer Calendrier comme table de dates

Vue Données → Calendrier → **Outils de table → Marquer comme table de dates → colonne Date**. Sans ça, `SAMEPERIODLASTYEAR` retournera blank silencieusement et toutes tes mesures de variation seront cassées.

---
# III — Créer la table `_Mesures`

**Accueil → Entrer des données →** 1 colonne, 1 ligne vide → nommer `_Mesures` → **Charger**.

Après avoir créé ta première mesure et l'avoir glissée dans `_Mesures`, supprime la colonne fictive : la table devient « table de mesures pure ».

---
# IV — Construire les 62 mesures DAX

### Vue d'ensemble des 8 dossiers

| Dossier | Mesures | Rôle |
|---|---|---|
| `Pédagogie` | 9 | Inscriptions, complétion, abandon, CSAT, qualité instructeurs |
| `Démographie` | 7 | Apprenants, % premium, distribution âges, pays |
| `Acquisition` | 2 | Canal d'acquisition + couleur |
| `Paiement` | 2 | Méthode de paiement + couleur |
| `Business` | 4 | Revenu FCFA / EUR, panier moyen, rang Top 5 |
| `Variations vs N-1` | 8 | Comparaisons N-1 + couleurs conditionnelles |
| `Top 5 Parcours` | 15 | Tops 1 à 5 (Nom, Revenu, Inscrits) |
| `Alerte Décrochage` | 15 | Compteurs ML, scores, actions recommandées |

## 4.1 Dossier `Pédagogie` (9 mesures)

```dax
Nb Inscriptions = COUNTROWS(inscriptions_analytics)
```

```dax
Taux Completion = 
VAR _termine = CALCULATE(COUNTROWS(inscriptions_analytics), inscriptions_analytics[statut] = "Termine")
VAR _total   = COUNTROWS(inscriptions_analytics)
RETURN DIVIDE(_termine, _total)
```

```dax
Taux Abandon = 
VAR _abandon = CALCULATE(COUNTROWS(inscriptions_analytics), inscriptions_analytics[statut] = "Abandonne")
VAR _total   = COUNTROWS(inscriptions_analytics)
RETURN DIVIDE(_abandon, _total)
```

```dax
CSAT Moyen = AVERAGE(inscriptions_analytics[csat])
Nb Inscriptions Instructeur = COUNTROWS(inscriptions_analytics)
```

### 📘 La matrice instructeur

Classer les instructeurs en 4 segments selon complétion (médiane 33,3 %) et CSAT (médiane 4,23) :

| | Complétion ≤ 33 % | Complétion > 33 % |
|---|---|---|
| **CSAT > 4.23** | Coach CSAT | ⭐ Top Performer |
| **CSAT ≤ 4.23** | À coacher | À accompagner |

```dax
Quadrant Instructeur = 
VAR _csat = [CSAT Moyen]
VAR _comp = [Taux Completion]
RETURN SWITCH(TRUE(),
    _comp > 0.333 && _csat > 4.23, "Top Performer",
    _comp > 0.333 && _csat <= 4.23, "A accompagner",
    _comp <= 0.333 && _csat > 4.23, "Coach CSAT",
    "A coacher"
)
```

### Trio « pire parcours du domaine »

```dax
Pire Parcours du Domaine = 
VAR _tbl = ADDCOLUMNS(VALUES(parcours[titre]), "@tx", [Taux Abandon])
VAR _maxTx = MAXX(_tbl, [@tx])
VAR _topRow = TOPN(1, FILTER(_tbl, [@tx] = _maxTx), [@tx], DESC)
RETURN MAXX(_topRow, parcours[titre])
```

```dax
Pire Parcours Taux = 
VAR _tbl = ADDCOLUMNS(VALUES(parcours[titre]), "@tx", [Taux Abandon])
RETURN MAXX(_tbl, [@tx])
```

```dax
Pire Parcours Label = 
VAR _nom = [Pire Parcours du Domaine]
VAR _tx  = [Pire Parcours Taux]
VAR _nom_court = SWITCH(TRUE(),
    _nom = "Machine Learning Applique", "Machine Learning",
    _nom = "React & Node.js Fullstack", "React & Node.js",
    _nom = "SEO & Growth Hacking",      "SEO & Growth",
    _nom = "Management de Projet",      "Mgmt Projet",
    _nom = "HTML CSS JavaScript",       "HTML CSS JS",
    _nom = "Python Web avec Django",    "Python Django",
    _nom = "Data Analyse avec Python",  "Data Python",
    _nom = "Power BI & Dashboards",     "Power BI",
    _nom = "SQL & Bases de Donnees",    "SQL & BDD",
    _nom = "Leadership & Soft Skills",  "Leadership",
    _nom = "Community Management",      "Community Mgmt",
    _nom = "Marketing Digital",         "Marketing Dig.",
    _nom
)
RETURN _nom_court & " " & FORMAT(_tx, "0.0%")
```

### ⚠️ `Quadrant Instructeur` ne peut PAS servir de légende dans un nuage de points (c'est une mesure). Pour le scatter, utilise la **colonne calculée** `parcours[Quadrant Instructeur Col]` créée en partie V.

## 4.2 Dossier `Démographie` (7 mesures)

```dax
Nb apprenants = DISTINCTCOUNT(apprenants[apprenant_id])
```

```dax
% Apprenants Premium = 
VAR _premium = CALCULATE(COUNTROWS(apprenants), apprenants[premium] = 1)
VAR _total   = COUNTROWS(apprenants)
RETURN DIVIDE(_premium, _total)
```

```dax
Inscriptions par Apprenant = 
DIVIDE(COUNTROWS(inscriptions_analytics), DISTINCTCOUNT(inscriptions_analytics[apprenant_id]))
```

```dax
% Distribution Ages = 
VAR _curr = CALCULATE(COUNTROWS(apprenants), apprenants[age] >= 18)
VAR _total = CALCULATE(
    COUNTROWS(apprenants),
    ALL(apprenants[Tranche Age]),
    ALL(apprenants[Ordre Tranche Age]),
    apprenants[age] >= 18
)
RETURN DIVIDE(_curr, _total)
```

```dax
Couleur Tranche Age = 
VAR _ordreCurr = SELECTEDVALUE(apprenants[Ordre Tranche Age])
RETURN IF(_ordreCurr IN { 2, 3 }, "#1FA67D", "#7C6FD9")
```
*Vert pour les tranches 25-34 et 35-44 ans (dominantes), violet pour les autres.*

### 📘 La table `Pays Affichage`

Table de référence statique à 6 lignes : Côte d'Ivoire, Sénégal, Cameroun, Mali, Burkina Faso, Autres. La mesure compte les apprenants nominaux des Top 5 OU regroupe le reste dans Autres.

```dax
% Apprenants par Pays = 
VAR _pays_courant = SELECTEDVALUE('Pays Affichage'[Pays Affichage])
VAR _total = CALCULATE(COUNTROWS(apprenants), ALL(apprenants))
VAR _top5 = { "Cote d'Ivoire", "Senegal", "Cameroun", "Mali", "Burkina Faso" }
VAR _nb = SWITCH(TRUE(),
    _pays_courant = "Autres",
        CALCULATE(COUNTROWS(apprenants), ALL(apprenants), NOT(apprenants[pays] IN _top5)),
    CALCULATE(COUNTROWS(apprenants), ALL(apprenants), apprenants[pays] = _pays_courant)
)
RETURN DIVIDE(_nb, _total)
```

```dax
Couleur Pays Affichage = SELECTEDVALUE('Pays Affichage'[Couleur])
```

## 4.3 Dossier `Acquisition` (2 mesures)

```dax
% Apprenants par Canal = 
VAR _canal_courant = SELECTEDVALUE(apprenants[canal_acquisition])
VAR _curr = CALCULATE(
    COUNTROWS(apprenants), ALL(apprenants),
    apprenants[canal_acquisition] = _canal_courant
)
VAR _total = CALCULATE(
    COUNTROWS(apprenants), ALL(apprenants),
    apprenants[canal_acquisition] <> BLANK()
)
RETURN DIVIDE(_curr, _total)
```

```dax
Couleur Canal Acquisition = 
SWITCH(SELECTEDVALUE(apprenants[canal_acquisition]),
    "Organique",       "#1FA67D",
    "Reseaux sociaux", "#5B5BD6",
    "Partenaire",      "#F2A93B",
    "Publicite",       "#E5494D",
    "#BDBDBD"
)
```

## 4.4 Dossier `Paiement` (2 mesures)

```dax
% Paiements par Methode = 
VAR _m = SELECTEDVALUE(paiements[methode])
VAR _curr  = CALCULATE(COUNTROWS(paiements), ALL(paiements), paiements[methode] = _m)
VAR _total = CALCULATE(COUNTROWS(paiements), ALL(paiements))
RETURN DIVIDE(_curr, _total)
```

```dax
Couleur Methode Paiement = 
SWITCH(SELECTEDVALUE(paiements[methode]),
    "Mobile Money",   "#1FA67D",
    "Carte bancaire", "#5B5BD6",
    "Virement",       "#F2A93B",
    "PayPal",         "#7A7A7A",
    "#BDBDBD"
)
```

## 4.5 Dossier `Business` (4 mesures)

```dax
Revenu Genere = SUM(paiements[montant_fcfa])
```
*Format : `#,0" FCFA"`*

```dax
Revenu EUR = DIVIDE([Revenu Genere], 655.957) / 1000000
```
*Format : `"€ "0.00"M"` — taux fixe BCEAO 1 EUR = 655,957 FCFA.*

```dax
Panier Moyen FCFA = DIVIDE([Revenu Genere], COUNTROWS(paiements))
```

```dax
Top 5 Parcours Rang = RANKX(ALL(parcours[titre]), [Revenu Genere], , DESC, DENSE)
```
*Donne le rang d'un parcours selon son revenu, avec mode `DENSE` pour ne pas sauter de rang sur égalité.*

## 4.6 Dossier `Variations vs N-1` (8 mesures)

### 📘 L'année précédente est dynamique grâce à `MAX(Calendrier[Annee]) - 1`. Tu n'as jamais besoin de modifier le DAX quand l'année change.

```dax
Variation Inscriptions % = 
VAR _curr = [Nb Inscriptions]
VAR _prev = CALCULATE([Nb Inscriptions], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct  = DIVIDE(_curr - _prev, _prev)
VAR _f    = FORMAT(_pct, "+0%;-0%")
VAR _annee_prev = MAX(Calendrier[Annee]) - 1
VAR _suffix = " vs " & _annee_prev
RETURN SWITCH(TRUE(),
    _pct > 0, UNICHAR(9650) & " " & _f & _suffix,
    _pct < 0, UNICHAR(9660) & " " & _f & _suffix,
    UNICHAR(9650) & " 0%" & _suffix
)
```

```dax
Variation Completion vs Cible = 
VAR _curr  = [Taux Completion]
VAR _cible = 0.40
VAR _delta = (_curr - _cible) * 100
VAR _f     = FORMAT(_delta, "+0;-0") & "pp vs cible 40%"
RETURN SWITCH(TRUE(),
    _delta > 0, UNICHAR(9650) & " " & _f,
    _delta < 0, UNICHAR(9660) & " " & _f,
    UNICHAR(9650) & " 0pp vs cible 40%"
)
```
*La complétion se compare à une **cible fixe (40 %)**, pas à N-1.*

```dax
Variation Abandon pp = 
VAR _curr  = [Taux Abandon]
VAR _prev  = CALCULATE([Taux Abandon], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _delta = (_curr - _prev) * 100
VAR _f     = FORMAT(_delta, "+0.0;-0.0") & "pp"
VAR _annee_prev = MAX(Calendrier[Annee]) - 1
VAR _suffix = " vs " & _annee_prev
RETURN SWITCH(TRUE(),
    _delta > 0, UNICHAR(9650) & " " & _f & _suffix,
    _delta < 0, UNICHAR(9660) & " " & _f & _suffix,
    UNICHAR(9650) & " 0pp" & _suffix
)
```

```dax
Variation CSAT = 
VAR _curr  = [CSAT Moyen]
VAR _prev  = CALCULATE([CSAT Moyen], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _delta = _curr - _prev
VAR _f     = FORMAT(_delta, "+0.0;-0.0")
VAR _annee_prev = MAX(Calendrier[Annee]) - 1
VAR _suffix = " vs " & _annee_prev
RETURN SWITCH(TRUE(),
    _delta > 0, UNICHAR(9650) & " " & _f & _suffix,
    _delta < 0, UNICHAR(9660) & " " & _f & _suffix,
    UNICHAR(9650) & " 0" & _suffix
)
```

### Les 4 couleurs (sémantique inversée pour Abandon)

```dax
Couleur Variation Inscriptions = 
VAR _pct = DIVIDE(
    [Nb Inscriptions] - CALCULATE([Nb Inscriptions], SAMEPERIODLASTYEAR(Calendrier[Date])),
    CALCULATE([Nb Inscriptions], SAMEPERIODLASTYEAR(Calendrier[Date]))
)
RETURN SWITCH(TRUE(), _pct > 0, "#1FA67D", _pct < 0, "#E5494D", "#9CA3AF")
```

```dax
Couleur Variation Completion = 
VAR _delta = [Taux Completion] - 0.40
RETURN SWITCH(TRUE(), _delta > 0, "#1FA67D", _delta < 0, "#E5494D", "#9CA3AF")
```

```dax
Couleur Variation Abandon = 
VAR _delta = [Taux Abandon] - CALCULATE([Taux Abandon], SAMEPERIODLASTYEAR(Calendrier[Date]))
RETURN SWITCH(TRUE(), _delta > 0, "#E5494D", _delta < 0, "#1FA67D", "#9CA3AF")
```
*Sens inversé : abandon en hausse = rouge.*

```dax
Couleur Variation CSAT = 
VAR _delta = [CSAT Moyen] - CALCULATE([CSAT Moyen], SAMEPERIODLASTYEAR(Calendrier[Date]))
RETURN SWITCH(TRUE(), _delta > 0, "#1FA67D", _delta < 0, "#E5494D", "#9CA3AF")
```

## 4.7 Dossier `Top 5 Parcours` (15 mesures)

### 📘 Pourquoi 15 mesures plutôt qu'un visuel Top N ?

Pour un dashboard où chaque card affiche **un Top N précis** (Top 1 à Top 5), Power BI ne sait pas afficher 5 cards distinctes via un simple visuel filtré. Solution : 5 séries de 3 mesures (`TopN Nom`, `TopN Revenu`, `TopN Inscrits`).

### Pattern pour le rang N (à dupliquer 5 fois)

```dax
Top1 Nom = 
VAR _r = ADDCOLUMNS(
    VALUES(parcours[titre]),
    "@rev",  [Revenu Genere],
    "@rang", RANKX(ALL(parcours[titre]), [Revenu Genere], , DESC, DENSE)
)
VAR _row = FILTER(_r, [@rang] = 1)
RETURN MAXX(_row, parcours[titre])
```

```dax
Top1 Revenu = 
VAR _r = ADDCOLUMNS(
    VALUES(parcours[titre]),
    "@rev",  [Revenu Genere],
    "@rang", RANKX(ALL(parcours[titre]), [Revenu Genere], , DESC, DENSE)
)
VAR _row = FILTER(_r, [@rang] = 1)
VAR _v   = MAXX(_row, [@rev])
RETURN SWITCH(TRUE(),
    _v >= 1000000, FORMAT(_v / 1000000, "0") & "M FCFA",
    FORMAT(_v / 1000, "0") & "k FCFA"
)
```
*Format dynamique : k FCFA en dessous d'1M, M FCFA au-dessus.*

```dax
Top1 Inscrits = 
VAR _r = ADDCOLUMNS(
    VALUES(parcours[titre]),
    "@rev",  [Revenu Genere],
    "@rang", RANKX(ALL(parcours[titre]), [Revenu Genere], , DESC, DENSE)
)
VAR _row = FILTER(_r, [@rang] = 1)
VAR _nom = MAXX(_row, parcours[titre])
VAR _nb  = CALCULATE(COUNTROWS(inscriptions_analytics), ALL(parcours), parcours[titre] = _nom)
RETURN FORMAT(_nb, "#,0") & " inscrits"
```

Pour `Top2`, `Top3`, `Top4`, `Top5` : duplique en remplaçant `[@rang] = 1` par `= 2`, `= 3`, `= 4`, `= 5`. **Total : 15 mesures.**

### ⚠️ `RANKX` sans `DENSE` : sur deux parcours à revenu identique (rare mais possible), tu sauterais des rangs (`1, 1, 3, 3, 5` au lieu de `1, 1, 2, 2, 3`) et `Top3 Nom` retournerait blank. Toujours `DENSE`.

## 4.8 Dossier `Alerte Décrochage` (15 mesures)

### 📘 Les 4 niveaux de risque ML

- **Rouge** (score > 0.80) — appel urgent
- **Orange** (0.60 ≤ score ≤ 0.80) — email perso
- **Violet** (0.30 ≤ score < 0.60) — push notification
- **Gris** (< 0.30) — RAS

### Compteurs (5 mesures)

```dax
Nb Apprenants Risque Total = COUNTROWS(apprenants_risque_scores)
Nb Apprenants Alerte = CALCULATE(COUNTROWS(apprenants_risque_scores), apprenants_risque_scores[alerte_decrochage] = 1)
Nb Apprenants Score > 0.80 = CALCULATE(COUNTROWS(apprenants_risque_scores), apprenants_risque_scores[score_risque] > 0.80)
```

```dax
Nb Apprenants Score 0.60 - 0.80 = 
CALCULATE(
    COUNTROWS(apprenants_risque_scores),
    apprenants_risque_scores[score_risque] >= 0.60,
    apprenants_risque_scores[score_risque] <= 0.80
)
```

```dax
Nb Apprenants Score 0.30 - 0.60 = 
CALCULATE(
    COUNTROWS(apprenants_risque_scores),
    apprenants_risque_scores[score_risque] >= 0.30,
    apprenants_risque_scores[score_risque] < 0.60
)
```

### Phrases dynamiques (2 mesures)

```dax
Phrase Alerte Decrochage = 
VAR _alerte = [Nb Apprenants Alerte]
VAR _total  = [Nb Apprenants Risque Total]
RETURN FORMAT(_alerte, "#,0") & " apprenants en alerte de decrochage sur " & FORMAT(_total, "#,0") & " en cours"
```

```dax
Taux Alerte Decrochage Label = 
VAR _taux = DIVIDE([Nb Apprenants Alerte], [Nb Apprenants Risque Total])
RETURN FORMAT(_taux, "0.0%") & " taux d'alerte"
```

### Labels par seuil (3 mesures)

```dax
Label Apprenants Score > 0.80      = [Nb Apprenants Score > 0.80] & " apprenants - Appel urgent"
Label Apprenants Score 0.60 - 0.80 = [Nb Apprenants Score 0.60 - 0.80] & " apprenants - Email perso"
Label Apprenants Score 0.30 - 0.60 = [Nb Apprenants Score 0.30 - 0.60] & " apprenants - Push"
```

### Action et couleurs (3 mesures)

```dax
Action Recommandee = 
VAR _score = SELECTEDVALUE(apprenants_risque_scores[score_risque])
RETURN SWITCH(TRUE(),
    _score > 0.80,                          "Appel urgent",
    _score >= 0.60 && _score <= 0.80,       "Email perso",
    _score >= 0.30 && _score < 0.60,        "Push",
    "RAS"
)
```

```dax
Couleur Action = 
SWITCH([Action Recommandee],
    "Appel urgent", "#E5494D",
    "Email perso",  "#F2A93B",
    "Push",         "#7C6FD9",
    "#9CA3AF"
)
```

```dax
Couleur Score Risque = 
VAR _score = SELECTEDVALUE(apprenants_risque_scores[score_risque])
RETURN SWITCH(TRUE(),
    _score > 0.80,  "#E5494D",
    _score >= 0.60, "#F2A93B",
    _score >= 0.30, "#7C6FD9",
    "#888888"
)
```

### Labels colonne pour le tableau actionnable (2 mesures)

```dax
Inactivite Label = 
VAR _val = MAX(apprenants_risque_scores[nb_jours_inactif])
RETURN IF(ISBLANK(_val), BLANK(), FORMAT(_val, "#,0") & "j")
```

```dax
Progression Label = 
VAR _val = MAX(apprenants_risque_scores[progression_pct])
RETURN IF(ISBLANK(_val), BLANK(), FORMAT(_val, "0") & "%")
```

### ⚠️ Pourquoi `MAX` plutôt que `SELECTEDVALUE` ?

Sur une cellule de tableau filtrée par apprenant, on a typiquement une seule ligne — `SELECTEDVALUE` et `MAX` renvoient la même valeur. Mais si pour une raison technique il y a plusieurs lignes, `SELECTEDVALUE` retourne BLANK silencieusement tandis que `MAX` reste exploitable. Plus robuste.

---
# V — Créer les colonnes calculées

### 📘 Règle de pouce — colonne calculée vs mesure

Si tu veux la valeur en **slicer, légende, axe**, c'est une **colonne calculée**. Sinon c'est une **mesure**.

## 5.1 `apprenants[Tranche Age]` & `[Ordre Tranche Age]`

```dax
Tranche Age = 
SWITCH(TRUE(),
    apprenants[age] < 25,                            "18-24 ans",
    apprenants[age] >= 25 && apprenants[age] < 35,   "25-34 ans",
    apprenants[age] >= 35 && apprenants[age] < 45,   "35-44 ans",
    apprenants[age] >= 45 && apprenants[age] < 55,   "45-54 ans",
    "55+ ans"
)
```

```dax
Ordre Tranche Age = 
SWITCH(apprenants[Tranche Age],
    "18-24 ans", 1,
    "25-34 ans", 2,
    "35-44 ans", 3,
    "45-54 ans", 4,
    "55+ ans",   5,
    99
)
```

### 🔧 Trier `Tranche Age` par `Ordre Tranche Age`

Sélectionne `Tranche Age` → **Outils de colonne → Trier par colonne → Ordre Tranche Age**.

## 5.2 `parcours[Quadrant Instructeur Col]`

Reproduit la logique de la mesure `[Quadrant Instructeur]` mais en colonne, pour pouvoir servir de légende dans le scatter.

```dax
Quadrant Instructeur Col = 
VAR _csat = CALCULATE(AVERAGE(inscriptions_analytics[csat]))
VAR _comp = 
    DIVIDE(
        CALCULATE(COUNTROWS(inscriptions_analytics), inscriptions_analytics[statut] = "Termine"),
        CALCULATE(COUNTROWS(inscriptions_analytics))
    )
RETURN SWITCH(TRUE(),
    _comp > 0.333 && _csat > 4.23,  "Top Performer",
    _comp > 0.333 && _csat <= 4.23, "A accompagner",
    _comp <= 0.333 && _csat > 4.23, "Coach CSAT",
    "A coacher"
)
```

### ⚠️ Le `CALCULATE` interne propage le contexte de ligne (le `parcours_id` courant) vers `inscriptions_analytics` grâce à la relation. Sans `CALCULATE`, l'agrégat se ferait sur toutes les inscriptions et chaque parcours aurait la même valeur.

## 5.3 Colonnes utilitaires

**Sur `apprenants_risque_scores`** :

```dax
Apprenant Nom Complet = apprenants_risque_scores[prenom] & " " & apprenants_risque_scores[nom]
```

```dax
Parcours Court = 
VAR _t = apprenants_risque_scores[titre]
RETURN SWITCH(TRUE(),
    _t = "Machine Learning Applique", "Machine Learning",
    _t = "React & Node.js Fullstack", "React & Node.js",
    _t = "Data Analyse avec Python",  "Data Python",
    _t = "Power BI & Dashboards",     "Power BI",
    _t = "SQL & Bases de Donnees",    "SQL & BDD",
    _t
)
```

**Sur `Calendrier`** :

```dax
Jour Semaine Nom   = FORMAT(Calendrier[Date], "dddd", "fr-FR")
Jour Semaine Ordre = WEEKDAY(Calendrier[Date], 2)
```

Trier `Jour Semaine Nom` par `Jour Semaine Ordre` pour que lundi → dimanche s'affichent dans le bon ordre.

---
# VI — Design system

## 6.1 Charte graphique EduTrack

EduTrack utilise une charte **light theme** avec accents violet et vert.

| Rôle | Hex | Usage |
|---|---|---|
| Primaire (violet) | `#7C6FD9` / `#5B5BD6` | Sidebar, accents, push notifications |
| Secondaire (vert) | `#1FA67D` | Variations positives, état OK |
| Warning | `#F2A93B` | Alertes orange, agents Coach |
| Danger | `#E5494D` | Alertes rouge, abandon en hausse |
| Neutre | `#9CA3AF` `#888888` `#7A7A7A` `#BDBDBD` | Variation à 0, axes, textes secondaires |
| Fond page | `#F9FAFB` | Arrière-plan rapport |
| Texte principal | `#111827` | Hero numbers |


## 6.2 Mockup PowerPoint → fonds PNG d'arrière-plan

### 📘 Concept clé

Power BI gère mal les arrière-plans complexes (cards à ombre, sidebars). La méthode pro :
1. Dessiner la mise en page dans **PowerPoint** (mockup vierge sans données)
2. Exporter en **PNG haute résolution** (1280×720 ou 2001×1125)
3. Importer comme **arrière-plan de page** dans Power BI
4. Poser les visuels Power BI **par-dessus**

### Ce que le mockup PPTX vierge doit contenir

✅ **À inclure :**
- Sidebar de navigation avec onglets (item actif surligné)
- Logo AfriCare + DataProjectLab Academy en footer
- Cards / rectangles vides avec border-top accent coloré
- Encadrés vides pour les charts et tableaux

❌ **À NE PAS inclure :**
- Titre de page (zone de texte Power BI dynamique)
- Slicers Année / Mois (segments natifs)
- Bouton « Retour à l'accueil » (bouton natif avec action Navigation)
- Toute valeur de KPI ou texte dans les cards
- Données dans les charts ou tableaux

### 🔧 Méthode 1 — Export PNG depuis PowerPoint (recommandé)

PowerPoint exporte par défaut en 96 DPI. Pour un 150 DPI lisible :

1. **Win + R** → `regedit` → **Entrée**
2. Aller dans `HKEY_CURRENT_USER\Software\Microsoft\Office\16.0\PowerPoint\Options`
3. Clic droit → **Nouveau** → **Valeur DWORD (32 bits)**
4. Nom : `ExportBitmapResolution` — Valeur : `150` (décimal)
5. Fermer regedit, **redémarrer PowerPoint**

Puis : **Fichier** → **Enregistrer sous** → **PNG** → **Toutes les diapositives**.

### 🔧 Méthode 2 — CloudConvert

1. Aller sur [cloudconvert.com/pptx-to-png](https://cloudconvert.com/pptx-to-png)
2. Charger `mockup_africare_blank.pptx`
3. Options → 150 DPI → 1280×720
4. **Convert** → télécharger les 4 PNG

### Renommage final

```
bg-00-home.png
bg-01-overview.png
bg-02-apprenants.png
bg-03-parcours.png
bg-04-revenus.png
bg-05-alerte-ml.png
```

### 🔧 Application dans Power BI

1. Sélectionner la page → **Format de la page** (icône pinceau au niveau page)
2. **Arrière-plan de la page** → **Ajouter une image** → choisir le PNG
3. **Ajustement de l'image** → **Adapter**
4. **Transparence** → **0 %**

Application : **Format de la page → Arrière-plan → Ajouter une image → Adapter → Transparence 0 %**.

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/powerbi/tuto/02_page.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# VII — Construire les 5 pages

> *« Pose les visuels par-dessus les fonds PNG en respectant la grille du mockup. La sidebar à gauche porte la navigation (Vue executive · Apprenants · Parcours · Revenus · Alertes ML). »*

## 7.1 Page 1 — Vue executive

> *« Quelle est la santé globale d'EduTrack ce trimestre ? »*

| Zone | Visuel | Champ / Mesure |
|---|---|---|
| Slicers (bas-gauche sidebar) | Liste horizontale | `Calendrier[Annee]` (2022, 2023, 2024) |
| KPI 1 | Carte | `Nb Inscriptions` + variation `Variation Inscriptions %` (couleur `Couleur Variation Inscriptions`) |
| KPI 2 | Carte | `Taux Completion` (vert) + `Variation Completion vs Cible` (couleur `Couleur Variation Completion`) |
| KPI 3 | Carte | `Taux Abandon` (rouge) + `Variation Abandon pp` (couleur `Couleur Variation Abandon`) |
| KPI 4 | Carte | `CSAT Moyen` (orange) + `Variation CSAT` (couleur `Couleur Variation CSAT`) |
| Évolution mensuelle | Courbes (axe Y double) | Axe X : `Calendrier[Annee_Mois]`, Y1 : `Nb Inscriptions` (violet), Y2 : `Taux Abandon` (rouge) |
| Répartition par domaine | Donut | `parcours[domaine]` × `Nb Inscriptions`, étiquettes en pourcentage |
| Top 10 parcours | Table triée descendant | `parcours[titre]`, `Taux Completion`, `Taux Abandon` (couleur conditionnelle vert / rouge) |
| Bouton retour | Texte cliquable | « Retour à l'accueil » → action Navigation vers la cover |

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/powerbi/tuto/screenshot_01.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 7.2 Page 2 — Apprenants

> *« Qui sont nos apprenants (géographie, âge, canal d'acquisition, paiement) ? »*

| Zone | Visuel | Champ / Mesure |
|---|---|---|
| KPI 1 | Carte | `Nb apprenants` (Apprenants actifs) |
| KPI 2 | Carte | `Inscriptions par Apprenant` |
| KPI 3 | Carte | `% Apprenants Premium` |
| Répartition géographique | Barres horizontales | `Pays Affichage[Pays Affichage]` × `% Apprenants par Pays`, couleur `Couleur Pays Affichage` (Top 5 pays + Autres) |
| Distribution des âges | Barres verticales avec data labels | `apprenants[Tranche Age]` × `% Distribution Ages`, couleur `Couleur Tranche Age` (vert pour 23-27 et 28-32 dominantes, violet sinon) |
| Funnel d'acquisition | Barres horizontales | `apprenants[canal_acquisition]` × `% Apprenants par Canal`, couleur `Couleur Canal Acquisition` |
| Méthodes de paiement | Barres horizontales | `paiements[methode]` × `% Paiements par Methode`, couleur `Couleur Methode Paiement` |

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/powerbi/tuto/screenshot_02.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 7.3 Page 3 — Parcours

> *« Qui sont les Top Performer / À accompagner / Coach CSAT / À coacher, et quel parcours décroche le plus dans chaque domaine ? »*

| Zone | Visuel | Champ / Mesure |
|---|---|---|
| Matrice performance instructeurs | Nuage de points (4 segments colorés) | X : `CSAT Moyen`, Y : `Taux Completion`, taille : `Nb Inscriptions Instructeur`, **légende : `parcours[Quadrant Instructeur Col]`** (colonne calculée) |
| Activité Jour × Mois | Matrice / heatmap | Lignes : `Calendrier[Jour Semaine Nom]` (Lun→Dim), Colonnes : `Calendrier[Mois_court]` (janv→déc), Valeur : `Nb Inscriptions`, fond conditionnel violet pâle |
| Card domaine 1 | Carte multi-rows | `parcours[domaine]` = "Data & IA" → `Taux Abandon` (border-top rouge), `Pire Parcours du Domaine` |
| Card domaine 2 | Carte multi-rows | `parcours[domaine]` = "Developpement Web" → idem (border-top orange) |
| Card domaine 3 | Carte multi-rows | `parcours[domaine]` = "Marketing Digital" → idem (border-top orange) |
| Card domaine 4 | Carte multi-rows | `parcours[domaine]` = "Management" → idem (border-top violet) |

### ⚠️ Piège fréquent — la légende du scatter

Une **mesure** (`[Quadrant Instructeur]`) ne peut pas servir de légende. C'est pour ça qu'on utilise la **colonne calculée** `parcours[Quadrant Instructeur Col]` créée en §5.2. Les 4 segments doivent ressortir en 4 couleurs distinctes : gris (À accompagner), vert (À coacher), orange (Coach CSAT), violet (Top Performer).

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/powerbi/tuto/screenshot_03.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 7.4 Page 4 — Revenus

> *« Quel parcours rapporte le plus, et comment se répartit le revenu par domaine et par méthode de paiement ? »*

| Zone | Visuel | Champ / Mesure |
|---|---|---|
| KPI 1 | Carte (border-top vert) | `Revenu Genere` (FCFA Revenue Total) |
| KPI 2 | Carte (border-top violet) | `Revenu EUR` (Equivalent Euro) |
| KPI 3 | Carte (border-top orange) | `Panier Moyen FCFA` |
| Revenus mensuels par domaine | Barres empilées | Axe X : `Calendrier[Mois_court]`, Y empilé : `Revenu Genere`, légende : `parcours[domaine]` |
| Par méthode | Barres horizontales | `paiements[methode]` × `Revenu Genere` (formaté en M FCFA), couleur `Couleur Methode Paiement` |
| Top 5 parcours par revenu | 5 cartes alignées horizontalement | Card 1 : `Top1 Revenu` + `Top1 Nom` + `Top1 Inscrits` (médaille or, border-top jaune) ; idem Top2 (violet), Top3 (orange), Top4, Top5 |

### 📘 Le rang dans la card

L'icône médaille à gauche du revenu (or, violet, jaune, puis chiffres 4 et 5) marque visuellement le rang. Tu peux soit utiliser l'icône Unicode 🏅 colorée, soit insérer une petite image PNG dans la card. Le revenu et le nombre d'inscrits viennent des mesures `TopN Revenu` et `TopN Inscrits`.

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/powerbi/tuto/screenshot_04.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 7.5 Page 5 — Alertes ML

> *« Quels apprenants en cours sont à risque, et quelle action prioritaire prendre ? »*

| Zone | Visuel | Champ / Mesure |
|---|---|---|
| Bandeau alerte | Zone de texte (fond blanc, border-top rouge) | Titre fixe « Alertes ML — Détection du décrochage » + `Phrase Alerte Decrochage` |
| Tag taux d'alerte | Carte (fond rouge `#E5494D`) | `Taux Alerte Decrochage Label` |
| Card 1 — Intervention humaine | Carte action (icône `!` rouge) | Titre fixe « Intervention humaine » · `Score > 0.80` · « Appel téléphonique » · `Label Apprenants Score > 0.80` (border-top rouge) |
| Card 2 — Relance ciblée | Carte action (icône `@` orange) | Titre fixe « Relance ciblée » · `Score 0.60–0.80` · « Email personnalisé » · `Label Apprenants Score 0.60 - 0.80` (border-top orange) |
| Card 3 — Rappel doux | Carte action (icône cloche violette) | Titre fixe « Rappel doux » · `Score 0.30–0.60` · « Notification push » · `Label Apprenants Score 0.30 - 0.60` (border-top violet) |
| Slicer secondaire | Liste verticale | `parcours[domaine]` (Data & IA, Développement Web, Management, Marketing Digital) avec cases à cocher |
| Tableau actionnable | Table triée par score décroissant | `Apprenant Nom Complet`, `Parcours Court`, `Progression Label`, `Inactivite Label`, `score_risque` (cellule fond `Couleur Score Risque`), `Action Recommandee` (badge fond `Couleur Action`) |

### 📘 Comment lire le tableau

Chaque ligne est un apprenant en cours. Le score est coloré (rouge / orange / violet / gris) selon la zone de risque. La colonne **Action** propose une recommandation prête à l'emploi (Appel urgent / Email perso / Push / RAS) que le tutorat peut traiter directement. Trie le tableau par score descendant pour traiter d'abord les plus à risque.

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/powerbi/tuto/screenshot_05.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# VIII — Slicers, navigation, finitions

**Slicers globaux** sur chaque page : `Calendrier[Annee]` (Liste horizontale) + `Calendrier[Trimestre]`. Clic droit → **Synchroniser les segments → cocher toutes les pages**.

**Bouton « Retour à l'accueil »** sur les pages 2-4 : Insérer → Boutons → Vide → 32×32 px → Format → Texte → caractère ← → Action **Navigation de page** vers *Pédagogie & Acquisition*.

---
# IX — Validation et livraison

## 9.1 Checklist de recette

**Modèle** : 7 tables, 5 relations actives, 0 bidirectionnelle · `Calendrier` marquée comme table de dates · Auto Date/Time désactivé.

**Mesures** : 62 dans `_Mesures` · 8 dossiers nommés · format défini · aucun `/` (toutes en `DIVIDE`).

**Colonnes calculées** : `Tranche Age` triée par `Ordre Tranche Age` · `Quadrant Instructeur Col` créée · `Jour Semaine Nom` triée par `Jour Semaine Ordre`.

**Pages** : 4 pages avec sous-titre dynamique · Slicers Année/Trimestre synchronisés · Bouton retour fonctionnel sur pages 2-4 · Couleurs conformes à la charte.

**Performance** : ouverture < 5 s · aucun visuel en erreur.

## 9.2 Pièges fréquents et solutions

| Symptôme | Cause | Correction |
|---|---|---|
| `SAMEPERIODLASTYEAR` renvoie blank | `Calendrier` non marquée comme table de dates | Outils de table → Marquer comme table de dates |
| `score_risque` plante en agrégat | String avec point décimal | Forcer en Decimal Number dans Power Query, ou `VALUE(SUBSTITUTE(...))` |
| Le scatter refuse `Quadrant Instructeur` | Une mesure ne peut pas être une légende | Utiliser `parcours[Quadrant Instructeur Col]` |
| `Top3 Nom` retourne blank inattendu | `RANKX` sans `DENSE` saute des rangs | Toujours `, , DESC, DENSE` |
| Tranches d'âge mal ordonnées | Tri alphabétique | Trier `Tranche Age` par `Ordre Tranche Age` |
| `Inactivite Label` ne s'affiche pas | `MAX` retourne BLANK pour les sans-risque | Le `IF(ISBLANK(...))` est volontaire — cellule vide |
| Donut Pays sans « Autres » | Table `Pays Affichage` non reliée | Pas besoin de relation : la mesure utilise `ALL(apprenants)` et `IN _top5` |
| Flèches ▲ ▼ en carré | Police sans Unicode | Forcer **Segoe UI Symbol** sur le champ |

## 9.3 Storytelling exécutif

1. **Page 1** : « +X % d'inscriptions ce trimestre, mais le taux de complétion reste à -Y pp de la cible 40 %. »
2. **Page 2** : « 2 parcours portent 40 % du revenu, mais le pire parcours du domaine affiche un taux d'abandon de Z % — risque de réputation. »
3. **Page 3** : « N apprenants en alerte ROUGE → appel urgent du tutorat aujourd'hui. »
4. **Page 4** : « Les 25-34 ans sont X % du parc, à cibler pour la communication. »
5. **Reco** : prioriser le coaching sur les instructeurs « À coacher ».

## 9.4 Annexes — Règles DAX universelles

1. Toujours `DIVIDE` (jamais `/`)
2. Pattern `VAR ... RETURN`
3. `MAX` ou `SELECTEDVALUE` selon le besoin (`MAX` plus robuste)
4. `RANKX` toujours avec `, DESC, DENSE`
5. Variations en pp = soustraction, pas `DIVIDE`
6. `_annee_prev = MAX(Calendrier[Annee]) - 1` au lieu de figer le suffixe

### Mapping mockup PPTX ↔ pages Power BI

| Slide PPTX | Background PNG | Page Power BI |
|---|---|---|
| 1 | `bg-01-pedagogie-acquisition.png` | Pédagogie & Acquisition |
| 2 | `bg-02-top-parcours-business.png` | Top 5 Parcours & Business |
| 3 | `bg-03-alerte-ml.png` | Alerte Décrochage ML |
| 4 | `bg-04-demographie.png` | Démographie |

---
<div style="background:#1E3A5F;padding:24px 32px;border-radius:10px;color:#FFFFFF;font-family:Georgia,serif;text-align:center;">
<div style="font-size:22px;font-weight:700;margin-bottom:6px;">EduTrack Analytics</div>
<div style="font-size:13px;color:#CBD5E0;font-family:'Segoe UI',sans-serif;"><b>DataProjectLab</b> — apprendre la data sur des cas concrets, structurés et orientés métier.</div>
</div>